# 02 — Feature Engineering

This notebook builds the full station×day feature matrix used to train and evaluate
the LightGBM model. It constructs the calendar via hybrid progressive densification,
computes cumulative net flow using hourly buckets, derives inventory bounds [L, U],
applies the NA-handling strategy, and produces lag and 7-day rolling average features.
The result is a clean, model-ready DataFrame written to `data/processed/`.

## Assumptions & Dependencies

- The raw parquet file (`divvy.parquet`) must exist at the path provided when prompted.
- `01_eda.ipynb` should be reviewed first so the schema is understood, but it does
  not write any files that this notebook depends on.
- Required packages: `duckdb`, `pandas`, `numpy` (see `requirements.txt`).

## Setup — load data path and connect DuckDB

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils import get_data_path, connect_duckdb
from src.features import (
    build_station_day_calendar,
    compute_inventory_bounds,
    add_lag_features,
    add_rolling_features,
)

file_path = get_data_path()
con = connect_duckdb(file_path)

## Stage 1 — Station×Day Calendar (Hybrid Progressive Densification)

Each station gets a row for every day from its **own first observed date** through
the dataset-wide maximum date. Using each station's own start date (rather than the
global minimum) prevents inflating zero-flow days for stations that had not yet
opened, which would corrupt rolling averages and lag features.

Cumulative net flow is computed using **hourly buckets** (`EXTRACT(hour FROM starttime/stoptime)`),
not individual trip timestamps. Hourly bucketing is required for consistency with
the original model training — trip-level timestamps produce finer-grained min/max
values that shift the [L, U] bounds.

In [ ]:
calendar_df = build_station_day_calendar(con)
print(calendar_df.shape)
calendar_df.head()

## Stage 2 — Inventory Bounds [L, U] and Prediction Target

The feasible starting-inventory interval for each station×day is:

```
L = clip(-min_cumulative_flow, 0, station_capacity_day)
U = clip(station_capacity_day - max_cumulative_flow, 0, station_capacity_day)
s_true = (L + U) / 2
```

If U < L (inversion), we fall back to L=0, U=station_capacity_day to retain the
row with a conservative feasible range rather than discarding it.

In [ ]:
calendar_df = compute_inventory_bounds(calendar_df)
print("Inversion count:", (calendar_df['U'] < calendar_df['L']).sum())
calendar_df[['L', 'U', 's_true']].describe()

## Stage 3 — NA Handling

Each column has a specific fill strategy with a rationale:

- `station_capacity_day`: forward fill within station — dock count is constant until
  a new observation says otherwise.
- `min_cumulative_flow`, `max_cumulative_flow`: fill with 0 — zero-trip days mean the
  curve never moved; this gives L=0, U=capacity (full range feasible).
- `temperature`, `events`: forward fill within station — weather doesn't become unknown
  just because nobody rode a bike.
- Rolling 7-day averages for temperature and inventory: 3-tier fallback
  (station → city-wide → global mean).
- `trips_departed_roll7`, `trips_arrived_roll7`: fill with 0 — no activity means zero.

In [ ]:
# Forward fill station_capacity_day and weather within station group
fill_cols = ['station_capacity_day', 'temperature', 'events']
calendar_df[fill_cols] = (
    calendar_df.groupby('station_id')[fill_cols]
    .transform(lambda s: s.ffill())
)

# Fill zero-trip flow columns with 0
calendar_df[['min_cumulative_flow', 'max_cumulative_flow']] = (
    calendar_df[['min_cumulative_flow', 'max_cumulative_flow']].fillna(0)
)

## Stage 4 — Rolling 7-Day Average Features

Rolling features use a 3-tier fallback: station-level mean → city-wide mean by date
→ global dataset mean. `trips_departed_roll7` and `trips_arrived_roll7` are filled
with 0 rather than city averages because zero activity in the past 7 days is the
correct value, not a proxy.

In [ ]:
calendar_df = add_rolling_features(calendar_df)
calendar_df.filter(like='_roll7').isnull().sum()

## Stage 5 — Lag Features and Train/Test Split

Lag features shift each selected column by 1 day within the station group.
The first row per station is dropped after shifting because there is no valid
prior-day context — filling with anything would introduce fabricated data.

Train/test split:
- **Train:** `trip_date < 2017-10-01`
- **Test:** `trip_date >= 2017-10-01`

Training is restricted to stations that appear in the test set.

In [ ]:
calendar_df = add_lag_features(calendar_df)

test_stations = calendar_df.loc[calendar_df['trip_date'] >= '2017-10-01', 'station_id'].unique()
calendar_df = calendar_df[calendar_df['station_id'].isin(test_stations)]

train_df = calendar_df[calendar_df['trip_date'] < '2017-10-01']
test_df  = calendar_df[calendar_df['trip_date'] >= '2017-10-01']

print("Train rows:", len(train_df), "| Test rows:", len(test_df))

## Save Processed Data

In [ ]:
train_df.to_parquet('../data/processed/train.parquet', index=False)
test_df.to_parquet('../data/processed/test.parquet', index=False)
print("Saved train.parquet and test.parquet to data/processed/")

## Summary

**Outputs produced by this notebook:**
- `data/processed/train.parquet` — training feature matrix (trip_date < 2017-10-01)
- `data/processed/test.parquet` — test feature matrix (trip_date >= 2017-10-01)

**What the next notebook expects:** `03_modeling.ipynb` reads both parquet files
from `data/processed/` and expects the 13 feature columns plus L, U, and s_true.